# 03 Feature Group and Algorithm Comparison

本 notebook 回答两个问题：

1. 哪一组输入特征更适合预测 turnover-derived segment？
2. 在同一组输入下，哪一种算法表现更稳？

比较矩阵：

- Model A: proxy-only
- Model B: raw financial fields
- Model C: raw + ratio / flag features
- Model D: proxy + raw + ratio / flag features

每组特征都分别训练三种模型：

- Logistic Regression
- Decision Tree
- HistGradientBoosting

任务：

- BB vs non_BB
- BB / SME / MidCorporate

注意：turnover 与 turnover-derived label 只作为监督学习标签使用，不进入任何特征组。


In [ ]:
import json
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)


def find_repo_root():
    env_root = os.environ.get("PROJECT_ROOT")
    starts = []
    if env_root:
        starts.append(Path(env_root))
    starts.append(Path.cwd())

    for start in starts:
        start = start.resolve()
        for p in [start, *start.parents]:
            if (p / ".git").exists() and (p / "00 Data Preparation + EDA").exists():
                return p
    raise FileNotFoundError(
        "Cannot find repository root. Start Jupyter from inside the cloned repo, "
        "or set PROJECT_ROOT to the repository root."
    )


def require_exists(path, label="path"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def first_existing(paths, label):
    paths = [Path(path) for path in paths]
    for path in paths:
        if path.exists():
            return path
    tried = "\n".join(f"- {path}" for path in paths)
    raise FileNotFoundError(f"Cannot find {label}. Tried:\n{tried}")


PROJECT_ROOT = find_repo_root()
FINANCIAL_DIR = PROJECT_ROOT / "00 Data Preparation + EDA" / "Financial Data"
SMALL_TEST_DIR = FINANCIAL_DIR / "Small Scale Data Test"
SAMPLE_COMPANIES_DIR = SMALL_TEST_DIR / "Sample Companies"
LOCAL_DATA_ROOT = Path(os.environ.get(
    "LOCAL_DATA_ROOT",
    r"E:\000硕士毕设\财务数据\Local Large Data",
))
ACCOUNTS_ZIP_DIR = first_existing(
    [
        LOCAL_DATA_ROOT / "Accounts Data_2025.1_2026.5",
        FINANCIAL_DIR / "Accounts Data_2025.1_2026.5",
    ],
    "accounts ZIP directory",
)
BULK_MATCH_DIR = SMALL_TEST_DIR / "2025全年bulk匹配"
MODEL_PREP_DIR = BULK_MATCH_DIR / "模型训练与数据处理"
MODEL_TRAIN_DIR = MODEL_PREP_DIR / "模型训练"

require_exists(FINANCIAL_DIR, "financial data directory")
require_exists(SMALL_TEST_DIR, "small scale test directory")
require_exists(ACCOUNTS_ZIP_DIR, "accounts ZIP directory")


BASE_DIR = MODEL_PREP_DIR
OUTPUT_DIR = MODEL_TRAIN_DIR / "三算法横向对比"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_T1 = BASE_DIR / "financial_features_candidate_latest_model_ready_t1.csv"
FEATURE_CONFIG = BASE_DIR / "model_feature_columns.json"

require_exists(INPUT_T1, "T1 model-ready file")
require_exists(FEATURE_CONFIG, "feature config")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOCAL_DATA_ROOT:", LOCAL_DATA_ROOT)
print("Input:", INPUT_T1)
print("Feature config:", FEATURE_CONFIG)
print("Output:", OUTPUT_DIR)


In [ ]:
df = pd.read_csv(INPUT_T1, dtype={"CompanyNumber": str})

with open(FEATURE_CONFIG, "r", encoding="utf-8") as f:
    feature_config = json.load(f)

required_cols = [
    "latest_observed_segment_from_turnover",
    "model_label_bb_vs_non_bb",
]

missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required label columns: {missing_required}")

df = df.dropna(subset=required_cols).copy()

print("Rows:", len(df))
print("\nBB vs non_BB counts:")
display(df["model_label_bb_vs_non_bb"].value_counts())
print("\nMulticlass counts:")
display(df["latest_observed_segment_from_turnover"].value_counts())

## Feature Group Definition

这里先做 A/B/C 三组 ablation，再增加 Model D 作为 all-features 对照：

- Model A 检验 proxy score 本身是否已经能表达公司规模。
- Model B 检验原始财务字段是否比 proxy 更直接。
- Model C 检验 ratio / flag 是否提供额外信息。
- Model D 检验 proxy 在 raw + ratio / flag 之外是否还有额外信息。

所有特征组都不包含 turnover、observed segment、model label。


In [ ]:
PROXY_FEATURES = [
    "financial_core_score",
    "financial_conservative_score",
    "financial_core_score_raw",
    "financial_conservative_score_raw",
    "available_core_balance_field_count",
    "available_proxy_field_count",
    "financial_data_quality_score",
    "financial_evidence_tier",
    "Accounts_AccountCategory",
    "account_category_group",
    "primary_sector",
]

RAW_FINANCIAL_FEATURES = [
    "current_assets",
    "net_assets_liabilities",
    "equity",
    "cash",
    "debtors",
    "creditors_total",
    "employees",
    "profit_loss",
    "borrowings",
    "Accounts_AccountCategory",
    "account_category_group",
    "primary_sector",
]

RATIO_FEATURES = [
    "cash_to_current_assets",
    "debtors_to_current_assets",
    "creditors_to_current_assets",
    "net_assets_to_current_assets",
    "equity_to_current_assets",
    "borrowings_to_current_assets",
    "profit_loss_to_current_assets",
    "log_current_assets_per_employee",
    "log_creditors_per_employee",
    "log_debtors_per_employee",
    "log_cash_per_employee",
    "log_profit_loss_per_employee",
]

STRUCTURE_FLAG_FEATURES = [
    "cash_heavy_flag",
    "debtor_heavy_flag",
    "creditor_heavy_flag",
    "asset_heavy_flag",
    "low_employee_high_asset_flag",
    "negative_equity_flag",
    "negative_net_assets_flag",
]

FEATURE_GROUPS_RAW = {
    "Model_A_proxy_only": PROXY_FEATURES,
    "Model_B_raw_financial_fields": RAW_FINANCIAL_FEATURES,
    "Model_C_raw_ratio_flag_features": RAW_FINANCIAL_FEATURES + RATIO_FEATURES + STRUCTURE_FLAG_FEATURES,
    "Model_D_all_proxy_raw_ratio_flag_features": PROXY_FEATURES + RAW_FINANCIAL_FEATURES + RATIO_FEATURES + STRUCTURE_FLAG_FEATURES,
}

LEAKAGE_PATTERNS = [
    "turnover",
    "observed_segment",
    "model_label",
    "label_available",
]

def clean_feature_list(cols):
    cleaned = []
    removed = []
    for col in cols:
        if col not in df.columns:
            removed.append((col, "missing"))
            continue
        col_l = col.lower()
        if any(pattern in col_l for pattern in LEAKAGE_PATTERNS):
            removed.append((col, "leakage_name"))
            continue
        if col not in cleaned:
            cleaned.append(col)
    return cleaned, removed


FEATURE_GROUPS = {}
removed_rows = []
for group_name, cols in FEATURE_GROUPS_RAW.items():
    cleaned, removed = clean_feature_list(cols)
    FEATURE_GROUPS[group_name] = cleaned
    for col, reason in removed:
        removed_rows.append({"feature_group": group_name, "feature": col, "reason": reason})

feature_group_summary = pd.DataFrame(
    [{"feature_group": k, "n_features": len(v), "features": ", ".join(v)} for k, v in FEATURE_GROUPS.items()]
)
display(feature_group_summary)

if removed_rows:
    print("Removed missing/leakage-like columns:")
    display(pd.DataFrame(removed_rows))

In [ ]:
NUMERIC_HINTS = set(feature_config.get("numeric_feature_columns", []))
BINARY_HINTS = set(feature_config.get("binary_feature_columns", []))
CATEGORICAL_HINTS = set(feature_config.get("categorical_feature_columns", []))


def one_hot_encoder_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def split_feature_types(data, feature_cols):
    numeric_cols = []
    binary_cols = []
    categorical_cols = []

    for col in feature_cols:
        if col in CATEGORICAL_HINTS or data[col].dtype == "object":
            categorical_cols.append(col)
        elif col in BINARY_HINTS:
            binary_cols.append(col)
        else:
            non_null = data[col].dropna()
            unique_vals = set(non_null.unique().tolist())
            if len(unique_vals) <= 2 and unique_vals.issubset({0, 1, 0.0, 1.0, True, False}):
                binary_cols.append(col)
            else:
                numeric_cols.append(col)

    return numeric_cols, binary_cols, categorical_cols


def make_preprocessor(data, feature_cols):
    numeric_cols, binary_cols, categorical_cols = split_feature_types(data, feature_cols)

    transformers = []
    if numeric_cols:
        transformers.append(
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), numeric_cols)
        )
    if binary_cols:
        transformers.append(
            ("bin", Pipeline([
                ("to_float", FunctionTransformer(lambda X: pd.DataFrame(X).replace({True: 1, False: 0}).astype("float64"), validate=False)),
                ("imputer", SimpleImputer(strategy="most_frequent")),
            ]), binary_cols)
        )
    if categorical_cols:
        transformers.append(
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", one_hot_encoder_dense()),
            ]), categorical_cols)
        )

    return ColumnTransformer(transformers=transformers, remainder="drop", verbose_feature_names_out=False)


def make_pipelines(data, feature_cols):
    preprocessor = make_preprocessor(data, feature_cols)
    return {
        "logistic_regression": Pipeline([
            ("preprocess", preprocessor),
            ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)),
        ]),
        "decision_tree_depth4": Pipeline([
            ("preprocess", clone(preprocessor)),
            ("model", DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, class_weight="balanced", random_state=42)),
        ]),
        "hist_gradient_boosting": Pipeline([
            ("preprocess", clone(preprocessor)),
            ("model", HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, max_leaf_nodes=31, random_state=42)),
        ]),
    }

In [ ]:
def probability_frame(pipe, X, classes):
    if hasattr(pipe, "predict_proba"):
        proba = pipe.predict_proba(X)
        return pd.DataFrame(proba, columns=[f"prob_{c}" for c in classes], index=X.index)
    return pd.DataFrame(index=X.index)


def add_prediction_probability_fields(out, label_col="predicted_label"):
    prob_cols = [c for c in out.columns if c.startswith("prob_")]
    if not prob_cols:
        out["predicted_label_probability"] = np.nan
        out["prediction_probability_margin"] = np.nan
        return out
    ordered = np.sort(out[prob_cols].to_numpy(dtype=float), axis=1)
    out["predicted_label_probability"] = ordered[:, -1]
    out["prediction_probability_margin"] = ordered[:, -1] - ordered[:, -2] if len(prob_cols) > 1 else np.nan
    return out


def binary_metrics(y_true, y_pred, y_score):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_BB": precision_score(y_true, y_pred, pos_label="BB", zero_division=0),
        "recall_BB": recall_score(y_true, y_pred, pos_label="BB", zero_division=0),
        "f1_BB": f1_score(y_true, y_pred, pos_label="BB", zero_division=0),
        "roc_auc_BB": roc_auc_score((y_true == "BB").astype(int), y_score) if y_score is not None else np.nan,
        "pr_auc_BB": average_precision_score((y_true == "BB").astype(int), y_score) if y_score is not None else np.nan,
    }


def multiclass_metrics(y_true, y_pred, proba, classes):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    try:
        y_bin = pd.get_dummies(pd.Categorical(y_true, categories=classes))
        out["roc_auc_ovr_weighted"] = roc_auc_score(y_bin, proba, average="weighted", multi_class="ovr")
    except Exception:
        out["roc_auc_ovr_weighted"] = np.nan
    return out


def cross_validate_manual(pipe, X, y, task, classes, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    rows = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        model = clone(pipe)
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        model.fit(X_train, y_train)
        pred = model.predict(X_valid)
        proba = model.predict_proba(X_valid)

        if task == "BB_vs_non_BB":
            class_list = list(model.named_steps["model"].classes_)
            bb_idx = class_list.index("BB")
            metric_row = binary_metrics(y_valid, pred, proba[:, bb_idx])
        else:
            metric_row = multiclass_metrics(y_valid, pred, proba, list(model.named_steps["model"].classes_))

        metric_row["fold"] = fold
        rows.append(metric_row)

    fold_df = pd.DataFrame(rows)
    summary = fold_df.drop(columns=["fold"]).agg(["mean", "std"]).T.reset_index()
    summary.columns = ["metric", "mean", "std"]
    return summary, fold_df

## Train / Test Split


In [ ]:
TRAIN_INDEX, TEST_INDEX = train_test_split(
    df.index,
    test_size=0.25,
    random_state=42,
    stratify=df["latest_observed_segment_from_turnover"],
)

print("Train rows:", len(TRAIN_INDEX))
print("Test rows:", len(TEST_INDEX))
print("Test segment distribution:")
display(df.loc[TEST_INDEX, "latest_observed_segment_from_turnover"].value_counts())

## Run Comparison Matrix


In [ ]:
bb_metric_rows = []
bb_cv_rows = []
bb_prediction_frames = []

multi_metric_rows = []
multi_cv_rows = []
multi_prediction_frames = []

tree_rule_outputs = {}
feature_importance_rows = []

id_cols = [
    "CompanyNumber",
    "CompanyName",
    "primary_sector",
    "Accounts_AccountCategory",
    "latest_observed_turnover",
    "latest_observed_segment_from_turnover",
    "model_label_bb_vs_non_bb",
    "financial_core_score",
    "financial_conservative_score",
    "financial_evidence_tier",
]
available_id_cols = [c for c in id_cols if c in df.columns]


def collect_linear_or_tree_importance(pipe, task, feature_group, model_name):
    model = pipe.named_steps["model"]
    preprocess = pipe.named_steps["preprocess"]
    try:
        names = preprocess.get_feature_names_out()
    except Exception:
        names = np.array([f"feature_{i}" for i in range(getattr(model, "n_features_in_", 0))])

    rows = []
    if hasattr(model, "coef_"):
        coefs = model.coef_
        classes = list(getattr(model, "classes_", []))
        if coefs.ndim == 1:
            coefs = coefs.reshape(1, -1)
        for class_idx, class_name in enumerate(classes[: coefs.shape[0]]):
            for feature, importance in zip(names, coefs[class_idx]):
                rows.append({
                    "task": task,
                    "feature_group": feature_group,
                    "model": model_name,
                    "importance_type": "coefficient",
                    "class": class_name,
                    "feature": feature,
                    "importance": importance,
                    "abs_importance": abs(importance),
                    "importance_std": np.nan,
                })
    elif hasattr(model, "feature_importances_"):
        for feature, importance in zip(names, model.feature_importances_):
            rows.append({
                "task": task,
                "feature_group": feature_group,
                "model": model_name,
                "importance_type": "tree_importance",
                "class": "overall",
                "feature": feature,
                "importance": importance,
                "abs_importance": abs(importance),
                "importance_std": np.nan,
            })
    return rows


def collect_permutation_importance(pipe, X_test, y_test, task, feature_group, model_name, scoring):
    try:
        result = permutation_importance(
            pipe,
            X_test,
            y_test,
            scoring=scoring,
            n_repeats=5,
            random_state=42,
            n_jobs=None,
        )
        return [
            {
                "task": task,
                "feature_group": feature_group,
                "model": model_name,
                "importance_type": f"permutation_{scoring}",
                "class": "overall",
                "feature": feature,
                "importance": mean_imp,
                "abs_importance": abs(mean_imp),
                "importance_std": std_imp,
            }
            for feature, mean_imp, std_imp in zip(X_test.columns, result.importances_mean, result.importances_std)
        ]
    except Exception as exc:
        print(f"Permutation importance skipped for {task} / {feature_group} / {model_name}: {exc}")
        return []


for feature_group, feature_cols in FEATURE_GROUPS.items():
    print(f"\n=== {feature_group}: {len(feature_cols)} features ===")
    X = df[feature_cols].copy()

    X_train = X.loc[TRAIN_INDEX]
    X_test = X.loc[TEST_INDEX]

    y_bb = df["model_label_bb_vs_non_bb"]
    y_bb_train = y_bb.loc[TRAIN_INDEX]
    y_bb_test = y_bb.loc[TEST_INDEX]

    y_multi = df["latest_observed_segment_from_turnover"]
    y_multi_train = y_multi.loc[TRAIN_INDEX]
    y_multi_test = y_multi.loc[TEST_INDEX]

    pipelines = make_pipelines(df, feature_cols)

    for model_name, pipe in pipelines.items():
        print(f"Running BB task: {feature_group} / {model_name}")
        fitted = clone(pipe)
        fitted.fit(X_train, y_bb_train)
        pred = fitted.predict(X_test)
        classes = list(fitted.named_steps["model"].classes_)
        prob_df = probability_frame(fitted, X_test, classes)
        bb_score = prob_df["prob_BB"].to_numpy() if "prob_BB" in prob_df.columns else None

        metric_row = binary_metrics(y_bb_test, pred, bb_score)
        metric_row.update({
            "task": "BB_vs_non_BB",
            "feature_group": feature_group,
            "model": model_name,
            "n_features": len(feature_cols),
            "train_rows": len(X_train),
            "test_rows": len(X_test),
        })
        bb_metric_rows.append(metric_row)

        cv_summary, _ = cross_validate_manual(pipe, X, y_bb, "BB_vs_non_BB", ["BB", "non_BB"])
        cv_summary["task"] = "BB_vs_non_BB"
        cv_summary["feature_group"] = feature_group
        cv_summary["model"] = model_name
        bb_cv_rows.append(cv_summary)

        pred_frame = df.loc[TEST_INDEX, available_id_cols].copy()
        pred_frame["prediction"] = pred
        pred_frame["predicted_label"] = pred
        pred_frame = pd.concat([pred_frame, prob_df], axis=1)
        pred_frame = add_prediction_probability_fields(pred_frame)
        pred_frame["predicted_bb_label"] = pred
        pred_frame["predicted_bb_probability"] = pred_frame.get("prob_BB", np.nan)
        pred_frame["predicted_non_bb_probability"] = pred_frame.get("prob_non_BB", np.nan)
        pred_frame["task"] = "BB_vs_non_BB"
        pred_frame["feature_group"] = feature_group
        pred_frame["model"] = model_name
        bb_prediction_frames.append(pred_frame)

        feature_importance_rows.extend(collect_linear_or_tree_importance(fitted, "BB_vs_non_BB", feature_group, model_name))
        if model_name == "hist_gradient_boosting":
            feature_importance_rows.extend(collect_permutation_importance(fitted, X_test, y_bb_test, "BB_vs_non_BB", feature_group, model_name, "balanced_accuracy"))

        if model_name == "decision_tree_depth4":
            try:
                names = fitted.named_steps["preprocess"].get_feature_names_out()
                tree_rule_outputs[(feature_group, "BB_vs_non_BB")] = export_text(fitted.named_steps["model"], feature_names=list(names))
            except Exception as exc:
                tree_rule_outputs[(feature_group, "BB_vs_non_BB")] = f"Could not export tree: {exc}"

        print(f"Running multiclass task: {feature_group} / {model_name}")
        fitted_multi = clone(pipe)
        fitted_multi.fit(X_train, y_multi_train)
        pred_multi = fitted_multi.predict(X_test)
        multi_classes = list(fitted_multi.named_steps["model"].classes_)
        prob_multi = probability_frame(fitted_multi, X_test, multi_classes)
        metric_multi = multiclass_metrics(y_multi_test, pred_multi, prob_multi.to_numpy(), multi_classes)
        metric_multi.update({
            "task": "multiclass_segment",
            "feature_group": feature_group,
            "model": model_name,
            "n_features": len(feature_cols),
            "train_rows": len(X_train),
            "test_rows": len(X_test),
        })
        multi_metric_rows.append(metric_multi)

        cv_multi, _ = cross_validate_manual(pipe, X, y_multi, "multiclass_segment", multi_classes)
        cv_multi["task"] = "multiclass_segment"
        cv_multi["feature_group"] = feature_group
        cv_multi["model"] = model_name
        multi_cv_rows.append(cv_multi)

        pred_multi_frame = df.loc[TEST_INDEX, available_id_cols].copy()
        pred_multi_frame["prediction"] = pred_multi
        pred_multi_frame["predicted_label"] = pred_multi
        pred_multi_frame["predicted_segment"] = pred_multi
        pred_multi_frame = pd.concat([pred_multi_frame, prob_multi], axis=1)
        pred_multi_frame = add_prediction_probability_fields(pred_multi_frame)
        pred_multi_frame["predicted_segment_probability"] = pred_multi_frame["predicted_label_probability"]
        pred_multi_frame["task"] = "multiclass_segment"
        pred_multi_frame["feature_group"] = feature_group
        pred_multi_frame["model"] = model_name
        multi_prediction_frames.append(pred_multi_frame)

        feature_importance_rows.extend(collect_linear_or_tree_importance(fitted_multi, "multiclass_segment", feature_group, model_name))
        if model_name == "hist_gradient_boosting":
            feature_importance_rows.extend(collect_permutation_importance(fitted_multi, X_test, y_multi_test, "multiclass_segment", feature_group, model_name, "balanced_accuracy"))

        if model_name == "decision_tree_depth4":
            try:
                names = fitted_multi.named_steps["preprocess"].get_feature_names_out()
                tree_rule_outputs[(feature_group, "multiclass_segment")] = export_text(fitted_multi.named_steps["model"], feature_names=list(names))
            except Exception as exc:
                tree_rule_outputs[(feature_group, "multiclass_segment")] = f"Could not export tree: {exc}"

## Save Outputs


In [ ]:
bb_metrics = pd.DataFrame(bb_metric_rows)
bb_cv = pd.concat(bb_cv_rows, ignore_index=True)
bb_predictions = pd.concat(bb_prediction_frames, ignore_index=True)

multi_metrics = pd.DataFrame(multi_metric_rows)
multi_cv = pd.concat(multi_cv_rows, ignore_index=True)
multi_predictions = pd.concat(multi_prediction_frames, ignore_index=True)

feature_importance = pd.DataFrame(feature_importance_rows)
if not feature_importance.empty:
    feature_importance = feature_importance.sort_values(
        ["task", "feature_group", "model", "abs_importance"],
        ascending=[True, True, True, False],
    )

output_paths = {
    "feature_group_summary": OUTPUT_DIR / "feature_group_definition_summary.csv",
    "bb_metrics": OUTPUT_DIR / "feature_group_algorithm_bb_vs_non_bb_metrics.csv",
    "bb_cv": OUTPUT_DIR / "feature_group_algorithm_bb_vs_non_bb_cross_validation.csv",
    "bb_predictions": OUTPUT_DIR / "feature_group_algorithm_bb_vs_non_bb_predictions.csv",
    "multi_metrics": OUTPUT_DIR / "feature_group_algorithm_multiclass_metrics.csv",
    "multi_cv": OUTPUT_DIR / "feature_group_algorithm_multiclass_cross_validation.csv",
    "multi_predictions": OUTPUT_DIR / "feature_group_algorithm_multiclass_predictions.csv",
    "feature_importance": OUTPUT_DIR / "feature_group_algorithm_feature_importance.csv",
    "run_summary": OUTPUT_DIR / "feature_group_algorithm_run_summary.json",
}

feature_group_summary.to_csv(output_paths["feature_group_summary"], index=False, encoding="utf-8-sig")
bb_metrics.to_csv(output_paths["bb_metrics"], index=False, encoding="utf-8-sig")
bb_cv.to_csv(output_paths["bb_cv"], index=False, encoding="utf-8-sig")
bb_predictions.to_csv(output_paths["bb_predictions"], index=False, encoding="utf-8-sig")
multi_metrics.to_csv(output_paths["multi_metrics"], index=False, encoding="utf-8-sig")
multi_cv.to_csv(output_paths["multi_cv"], index=False, encoding="utf-8-sig")
multi_predictions.to_csv(output_paths["multi_predictions"], index=False, encoding="utf-8-sig")
feature_importance.to_csv(output_paths["feature_importance"], index=False, encoding="utf-8-sig")

for (feature_group, task), rules in tree_rule_outputs.items():
    safe_group = feature_group.replace("/", "_").replace(" ", "_")
    path = OUTPUT_DIR / f"feature_group_{safe_group}_{task}_decision_tree_rules.txt"
    path.write_text(rules, encoding="utf-8")

run_summary = {
    "input_t1": str(INPUT_T1),
    "feature_config": str(FEATURE_CONFIG),
    "output_dir": str(OUTPUT_DIR),
    "rows": int(len(df)),
    "feature_groups": FEATURE_GROUPS,
    "algorithms": ["logistic_regression", "decision_tree_depth4", "hist_gradient_boosting"],
    "tasks": ["BB_vs_non_BB", "multiclass_segment"],
    "bb_vs_non_bb_counts": df["model_label_bb_vs_non_bb"].value_counts().to_dict(),
    "multiclass_counts": df["latest_observed_segment_from_turnover"].value_counts().to_dict(),
    "outputs": {k: str(v) for k, v in output_paths.items()},
}

output_paths["run_summary"].write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved outputs:")
for name, path in output_paths.items():
    print(name, "->", path)

## BB vs non_BB: Model Comparison


In [ ]:
bb_display_cols = [
    "feature_group",
    "model",
    "n_features",
    "balanced_accuracy",
    "precision_BB",
    "recall_BB",
    "f1_BB",
    "roc_auc_BB",
    "pr_auc_BB",
    "accuracy",
]
display(
    bb_metrics[bb_display_cols]
    .sort_values(["balanced_accuracy", "f1_BB"], ascending=False)
    .reset_index(drop=True)
)

## BB vs non_BB: Cross Validation


In [ ]:
display(
    bb_cv.pivot_table(
        index=["feature_group", "model"],
        columns="metric",
        values="mean",
        aggfunc="first",
    )
    .reset_index()
    .sort_values("balanced_accuracy", ascending=False)
)

## Multiclass: Model Comparison


In [ ]:
multi_display_cols = [
    "feature_group",
    "model",
    "n_features",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "roc_auc_ovr_weighted",
    "accuracy",
]
display(
    multi_metrics[multi_display_cols]
    .sort_values(["balanced_accuracy", "macro_f1"], ascending=False)
    .reset_index(drop=True)
)

## Multiclass: Cross Validation


In [ ]:
display(
    multi_cv.pivot_table(
        index=["feature_group", "model"],
        columns="metric",
        values="mean",
        aggfunc="first",
    )
    .reset_index()
    .sort_values("balanced_accuracy", ascending=False)
)

## Confusion Matrices


In [ ]:
for (feature_group, model), g in bb_predictions.groupby(["feature_group", "model"]):
    print(f"\nBB task: {feature_group} / {model}")
    labels = ["BB", "non_BB"]
    cm = pd.DataFrame(
        confusion_matrix(g["model_label_bb_vs_non_bb"], g["predicted_bb_label"], labels=labels),
        index=[f"true_{x}" for x in labels],
        columns=[f"pred_{x}" for x in labels],
    )
    display(cm)

for (feature_group, model), g in multi_predictions.groupby(["feature_group", "model"]):
    print(f"\nMulticlass task: {feature_group} / {model}")
    labels = ["BB", "SME", "MidCorporate"]
    cm = pd.DataFrame(
        confusion_matrix(g["latest_observed_segment_from_turnover"], g["predicted_segment"], labels=labels),
        index=[f"true_{x}" for x in labels],
        columns=[f"pred_{x}" for x in labels],
    )
    display(cm)

## Feature Importance


In [ ]:
for (task, feature_group, model), g in feature_importance.groupby(["task", "feature_group", "model"]):
    print(f"\n{task} / {feature_group} / {model}")
    display(g[["importance_type", "class", "feature", "importance", "abs_importance", "importance_std"]].head(15))

## How to Read This Notebook

优先判断顺序：

1. 先比较同一算法下 Model A/B/C/D 的差异，判断哪组输入最有信息量。
2. 再比较同一特征组下三种算法的差异，判断是否存在明显的非线性收益。
3. 如果 Model C 明显优于 Model B，说明 ratio / flag features 有增量价值。
4. 如果 Model D 明显优于 Model C，说明 proxy 在 raw + ratio / flag 之外仍有增量价值。
5. 如果 Model A 接近 Model C/D，说明 proxy score 已经捕捉了大部分规模信息。
6. 如果 HGB 明显优于 Logistic Regression，说明 segment 与财务字段之间可能存在非线性关系。
